# Qwen3.5-27B-FineTree-V2.1 Colab Inference + Benchmark Bundle

This notebook is for the one-time backfill case where the model is already trained and only inference plus benchmark bundle generation is needed.

It runs on Colab with no dependency on the FineTree repo.

Workflow:

1. install dependencies
2. fetch the exact historical `args.json` from the model repo
3. fetch or upload historical `logging.jsonl`
4. configure the inference target and GPU batch size
5. run dataset-wide `transformers` inference on GPU
6. emit a self-contained benchmark bundle: `info.json`, `args.json`, `logging.jsonl`, `predictions/`, and a zip archive

Supported inference targets:

- local adapter
- remote adapter
- local full model
- remote full model

Default target:

- remote full model `asafd60/Qwen3.5-27B-FineTree-V2.1`


In [ ]:
# Exact dependency flow from the source notebook, minus flash-attention.
!pip install --upgrade pip
!python -m pip install -U uv
!uv pip install --system -U transformers trl peft datasets accelerate bitsandbytes huggingface_hub pillow qwen_vl_utils
!uv pip install ms-swift==v4.0.0


In [ ]:
import json
import os
import platform
import shlex
import shutil
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
from zoneinfo import ZoneInfo

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def israel_now_iso():
    return datetime.now(ZoneInfo("Asia/Jerusalem")).isoformat()

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def run_command(label, command, env_overrides=None):
    env = dict(os.environ)
    env.update(env_overrides or {})
    rendered = " ".join(shlex.quote(str(part)) for part in command)
    print(f"\n[{label}] {rendered}")
    process = subprocess.Popen(
        [str(part) for part in command],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-200:]).strip()
        raise RuntimeError(f"{label} failed with exit code {return_code}. Last output:\n{tail}")

def cli_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def append_cli_args(command, arg_map):
    out = list(command)
    for key, value in arg_map.items():
        if value is None:
            continue
        cli_key = "val_dataset" if key == "validation_dataset" else key
        out.extend([f"--{cli_key}", cli_value(value)])
    return out

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return path

def copy_hf_file(repo_id, filename, destination, token=None, required=True):
    from huggingface_hub import hf_hub_download
    try:
        downloaded = hf_hub_download(repo_id=repo_id, filename=filename, token=token)
    except Exception as exc:
        if required:
            raise
        print(f"Could not download {filename} from {repo_id}: {exc}")
        return None
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(downloaded, destination)
    return destination

def collect_environment_snapshot(active_env):
    gpu_names = []
    gpu_count = 0
    torch_version = None
    cuda_runtime_version = None
    try:
        import torch
        torch_version = getattr(torch, "__version__", None)
        cuda_runtime_version = getattr(getattr(torch, "version", None), "cuda", None)
        if torch.cuda.is_available():
            gpu_count = int(torch.cuda.device_count())
            gpu_names = [str(torch.cuda.get_device_name(i)) for i in range(gpu_count)]
    except Exception:
        pass
    nvidia_smi = None
    try:
        completed = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            check=False,
            capture_output=True,
            text=True,
        )
        if completed.returncode == 0:
            nvidia_smi = completed.stdout.strip() or None
            if not gpu_names and nvidia_smi:
                gpu_names = [line.split(",", 1)[0].strip() for line in nvidia_smi.splitlines() if line.strip()]
                gpu_count = len(gpu_names)
    except Exception:
        pass
    return {
        "platform": platform.platform(),
        "platform_machine": platform.machine(),
        "python_version": platform.python_version(),
        "python_executable": sys.executable,
        "hostname": socket.gethostname(),
        "torch_version": torch_version,
        "cuda_runtime_version": cuda_runtime_version,
        "ms_swift_version": package_version("ms-swift") or package_version("swift"),
        "transformers_version": package_version("transformers"),
        "cuda_visible_devices": active_env.get("CUDA_VISIBLE_DEVICES"),
        "max_pixels_env": active_env.get("MAX_PIXELS"),
        "gpu_count": gpu_count,
        "gpu_names": gpu_names,
        "gpu_used": " | ".join(gpu_names) if gpu_names else None,
        "nvidia_smi": nvidia_smi,
    }

def maybe_parse_json_string(text):
    stripped = str(text).strip()
    if not stripped.startswith("{"):
        return None
    try:
        payload = json.loads(stripped)
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None

def maybe_extract_json_object(text):
    if not isinstance(text, str):
        return None
    parsed = maybe_parse_json_string(text)
    if parsed is not None:
        return parsed
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        payload = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None

def extract_prediction_payload(row):
    if isinstance(row, dict):
        if isinstance(row.get("pages"), list) or isinstance(row.get("facts"), list):
            return row
        for key in ("prediction_json", "parsed_prediction", "prediction", "response", "generated_text", "output", "text"):
            value = row.get(key)
            if isinstance(value, dict) and (isinstance(value.get("pages"), list) or isinstance(value.get("facts"), list)):
                return value
            parsed = maybe_extract_json_object(value)
            if parsed is not None:
                return parsed
        messages = row.get("messages")
        if isinstance(messages, list):
            for message in reversed(messages):
                if not isinstance(message, dict):
                    continue
                parsed = maybe_extract_json_object(message.get("content"))
                if parsed is not None:
                    return parsed
    if isinstance(row, str):
        parsed = maybe_extract_json_object(row)
        if parsed is not None:
            return parsed
    raise ValueError("Could not extract a benchmark JSON payload from inference output. Refusing to fall back to labels or other non-prediction fields.")

def materialize_prediction_json_files(result_jsonl_path, output_dir, prefix="pred_"):
    result_jsonl_path = Path(result_jsonl_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    written_paths = []
    for index, line in enumerate(result_jsonl_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        payload = json.loads(line)
        prediction = extract_prediction_payload(payload)
        out_path = output_dir / f"{prefix}{index:04d}.json"
        out_path.write_text(json.dumps(prediction, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        written_paths.append(out_path)
    if not written_paths:
        raise ValueError(f"No prediction files were written from {result_jsonl_path}")
    return written_paths

def first_or_value(value):
    if isinstance(value, list) and len(value) == 1:
        return value[0]
    return value

def normalize_metadata_value(value):
    value = first_or_value(value)
    if isinstance(value, list):
        return ", ".join(str(item) for item in value)
    return value

BENCHMARK_METADATA_FIELDS = [
    "checkpoint_name", "model", "tuner_type", "dataset", "validation_dataset", "use_hf", "freeze_vit",
    "vit_lr", "freeze_aligner", "enable_thinking", "add_non_thinking_prefix", "torch_dtype",
    "num_train_epochs", "per_device_train_batch_size", "per_device_eval_batch_size", "learning_rate",
    "lora_rank", "lora_alpha", "target_modules", "gradient_accumulation_steps", "output_dir",
    "eval_steps", "save_steps", "save_total_limit", "logging_steps", "max_length", "warmup_ratio",
    "dataset_num_proc", "dataloader_num_workers", "eval_on_start", "max_pixels", "temperature",
    "gradient_checkpointing", "gradient_checkpointing_kwargs", "truncation_strategy",
    "CUDA_VISIBLE_DEVICES", "MAX_PIXELS", "gpu_used", "torch_env_used", "platform"
]

def build_benchmark_model_metadata(training_args, environment, checkpoint_name, metadata_overrides=None):
    metadata_out = {key: None for key in BENCHMARK_METADATA_FIELDS}
    for key in BENCHMARK_METADATA_FIELDS:
        if key in training_args:
            metadata_out[key] = normalize_metadata_value(training_args.get(key))
    metadata_out["checkpoint_name"] = checkpoint_name
    metadata_out["dataset"] = normalize_metadata_value(training_args.get("dataset"))
    metadata_out["validation_dataset"] = normalize_metadata_value(training_args.get("validation_dataset") or training_args.get("val_dataset"))
    metadata_out["target_modules"] = normalize_metadata_value(training_args.get("target_modules"))
    metadata_out["CUDA_VISIBLE_DEVICES"] = training_args.get("CUDA_VISIBLE_DEVICES") or environment.get("cuda_visible_devices")
    metadata_out["MAX_PIXELS"] = training_args.get("MAX_PIXELS") or environment.get("max_pixels_env")
    metadata_out["gpu_used"] = environment.get("gpu_used")
    metadata_out["torch_env_used"] = " | ".join(
        value for value in [
            f"torch {environment.get('torch_version')}" if environment.get("torch_version") else None,
            f"cuda {environment.get('cuda_runtime_version')}" if environment.get("cuda_runtime_version") else None,
            f"python {environment.get('python_version')}" if environment.get("python_version") else None,
        ]
        if value
    ) or None
    metadata_out["platform"] = environment.get("platform")
    if metadata_overrides:
        metadata_out.update(dict(metadata_overrides))
    return metadata_out

def build_submission_info_payload(model_metadata, training_args, environment, run, selected_checkpoint, artifacts):
    return {
        "model_metadata": dict(model_metadata),
        "training_args": dict(training_args),
        "environment": dict(environment),
        "run": dict(run),
        "selected_checkpoint": dict(selected_checkpoint),
        "artifacts": dict(artifacts),
    }

def _reference_label(value):
    text = str(value).rstrip("/")
    if not text:
        return "model"
    return text.split("/")[-1]

def resolve_inference_target(target_config, base_model):
    config = dict(target_config or {})
    mode = str(config.get("mode") or "remote_full_model").strip()
    if mode == "local_adapter":
        adapter_ref = config.get("adapter")
        if not adapter_ref:
            raise ValueError("local_adapter mode requires a local adapter path")
        model_ref = config.get("base_model") or base_model
        use_hf = bool(config.get("use_hf", True))
        label = str(config.get("label") or Path(str(adapter_ref)).name)
    elif mode == "remote_adapter":
        adapter_ref = config.get("adapter")
        if not adapter_ref:
            raise ValueError("remote_adapter mode requires an adapter repo id")
        model_ref = config.get("base_model") or base_model
        use_hf = True if config.get("use_hf") is None else bool(config.get("use_hf"))
        label = str(config.get("label") or _reference_label(adapter_ref))
    elif mode == "local_full_model":
        model_ref = config.get("model")
        if not model_ref:
            raise ValueError("local_full_model mode requires a local model path")
        adapter_ref = None
        use_hf = False if config.get("use_hf") is None else bool(config.get("use_hf"))
        label = str(config.get("label") or Path(str(model_ref)).name)
    elif mode == "remote_full_model":
        model_ref = config.get("model")
        if not model_ref:
            raise ValueError("remote_full_model mode requires a remote model repo id")
        adapter_ref = None
        use_hf = True if config.get("use_hf") is None else bool(config.get("use_hf"))
        label = str(config.get("label") or _reference_label(model_ref))
    else:
        raise ValueError("Unsupported inference target mode: " + mode)
    return {
        "mode": mode,
        "label": label,
        "use_hf": use_hf,
        "model_ref": str(model_ref),
        "adapter_ref": None if adapter_ref is None else str(adapter_ref),
        "base_model": str(base_model),
    }

def resolve_torch_dtype(dtype_name):
    import torch
    mapping = {
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
        "float16": torch.float16,
        "fp16": torch.float16,
        "float32": torch.float32,
        "fp32": torch.float32,
    }
    return mapping.get(str(dtype_name or "bfloat16").lower(), torch.bfloat16)

def load_inference_dataset(dataset_id):
    from datasets import load_dataset
    dataset = load_dataset(dataset_id)
    for split_name in ("train", "validation", "test"):
        if split_name in dataset:
            return dataset[split_name]
    raise RuntimeError(f"Dataset {dataset_id} does not expose a usable split")

def extract_sample_images(sample):
    images = sample.get("images")
    if images is None:
        image = sample.get("image")
        images = [] if image is None else [image]
    elif not isinstance(images, list):
        images = [images]
    if not images:
        raise ValueError("Inference sample is missing image content")
    return images

def build_inference_messages(sample, images):
    messages = sample.get("messages")
    if isinstance(messages, list) and messages:
        rebuilt = []
        image_index = 0
        for message in messages:
            if not isinstance(message, dict):
                continue
            role = str(message.get("role") or "user")
            if role == "assistant":
                continue
            content = message.get("content")
            if isinstance(content, str):
                rebuilt.append({"role": role, "content": [{"type": "text", "text": content}]})
                continue
            if not isinstance(content, list):
                continue
            rebuilt_content = []
            for item in content:
                if not isinstance(item, dict):
                    continue
                item_type = item.get("type")
                if item_type == "image":
                    if image_index >= len(images):
                        raise ValueError("Message image placeholders exceed available images")
                    rebuilt_content.append({"type": "image", "image": images[image_index]})
                    image_index += 1
                elif item_type == "text":
                    rebuilt_content.append({"type": "text", "text": item.get("text", "")})
            if rebuilt_content:
                rebuilt.append({"role": role, "content": rebuilt_content})
        if rebuilt:
            if image_index < len(images):
                for message in rebuilt:
                    if message.get("role") == "user":
                        for image in images[image_index:]:
                            message["content"].insert(0, {"type": "image", "image": image})
                        image_index = len(images)
                        break
            return rebuilt
    instruction = sample.get("instruction") or sample.get("prompt") or ""
    system_message = sample.get("system")
    fallback = []
    if system_message:
        fallback.append({"role": "system", "content": [{"type": "text", "text": system_message}]})
    fallback.append({
        "role": "user",
        "content": [{"type": "image", "image": images[0]}, {"type": "text", "text": str(instruction)}],
    })
    return fallback

def run_transformers_dataset_inference(target, dataset_id, raw_predictions_path, output_dir, inference_config):
    import torch
    from peft import PeftModel
    from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for transformers inference")

    dtype = resolve_torch_dtype(inference_config.get("torch_dtype"))
    processor_ref = target["base_model"] if target.get("adapter_ref") else target["model_ref"]
    processor_kwargs = {"trust_remote_code": True}
    max_pixels = inference_config.get("max_pixels")
    if max_pixels not in (None, ""):
        processor_kwargs["min_pixels"] = int(max_pixels)
        processor_kwargs["max_pixels"] = int(max_pixels)
    processor = AutoProcessor.from_pretrained(processor_ref, **processor_kwargs)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        target["base_model"] if target.get("adapter_ref") else target["model_ref"],
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    if target.get("adapter_ref"):
        model = PeftModel.from_pretrained(model, target["adapter_ref"])
    model.eval()

    eval_split = load_inference_dataset(dataset_id)
    raw_predictions_path = Path(raw_predictions_path)
    output_dir = Path(output_dir)
    raw_predictions_path.parent.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    for existing in output_dir.glob("pred_*.json"):
        existing.unlink()
    if raw_predictions_path.exists():
        raw_predictions_path.unlink()

    tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
    batch_size = int(inference_config.get("per_device_eval_batch_size") or 1)
    max_new_tokens = int(inference_config.get("max_new_tokens") or 2048)
    enable_thinking = bool(inference_config.get("enable_thinking", False))
    written_paths = []

    with raw_predictions_path.open("w", encoding="utf-8") as raw_handle:
        prediction_index = 0
        for batch_start in range(0, len(eval_split), batch_size):
            batch_stop = min(batch_start + batch_size, len(eval_split))
            batch_samples = [eval_split[index] for index in range(batch_start, batch_stop)]
            chat_texts = []
            batch_images = []
            batch_metadata = []
            for sample in batch_samples:
                sample_images = extract_sample_images(sample)
                if len(sample_images) != 1:
                    raise ValueError("Only single-image samples are supported in this notebook")
                messages = build_inference_messages(sample, sample_images)
                chat_text = processor.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=enable_thinking,
                )
                chat_texts.append(chat_text)
                batch_images.append(sample_images[0])
                batch_metadata.append({
                    "document_id": sample.get("document_id"),
                    "annotation_file": sample.get("annotation_file"),
                })

            inputs = processor(text=chat_texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in inputs.items()}
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
            prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
            for row_index, prompt_length in enumerate(prompt_lengths):
                prediction_index += 1
                output_tokens = generated[row_index, int(prompt_length):]
                response_text = tokenizer.decode(output_tokens, skip_special_tokens=True).strip()
                prediction = extract_prediction_payload({"response": response_text})
                prediction_path = output_dir / f"pred_{prediction_index:04d}.json"
                prediction_path.write_text(json.dumps(prediction, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
                raw_payload = {
                    "sample_index": prediction_index,
                    "response": response_text,
                    "prediction_file": prediction_path.name,
                    **batch_metadata[row_index],
                }
                raw_handle.write(json.dumps(raw_payload, ensure_ascii=False) + "\n")
                raw_handle.flush()
                written_paths.append(prediction_path)
                print(f"[prediction] wrote {prediction_path.name}")
            print(f"[batch] completed {batch_stop}/{len(eval_split)}")
    if not written_paths:
        raise ValueError(f"No prediction files were written for dataset {dataset_id}")
    return written_paths


In [ ]:
MODEL_REPO = "asafd60/Qwen3.5-27B-FineTree-V2.1"
DEFAULT_BASE_MODEL = "Qwen/Qwen3.5-27B"
HF_TOKEN = os.getenv("HF_TOKEN")

SWIFT_ENV = {
    "CUDA_VISIBLE_DEVICES": "0",
    "MAX_PIXELS": "1000000",
}

# Increase this if your GPU has enough memory.
INFERENCE_BATCH_SIZE = 1
INFERENCE_MAX_NEW_TOKENS = 12000

RUN_ID = f"qwen35-finetree-v21-infer-{time.strftime('%Y%m%d-%H%M%S')}"
COLAB_ROOT = Path("/content")
RUN_OUTPUT_DIR = COLAB_ROOT / "output" / "Qwen3.5-27B-FineTree-V2.1" / RUN_ID
BENCHMARK_BUNDLE_DIR = RUN_OUTPUT_DIR / "benchmark_submission"
PREDICTIONS_DIR = BENCHMARK_BUNDLE_DIR / "predictions"
RAW_PREDICTIONS_PATH = BENCHMARK_BUNDLE_DIR / "raw_predictions.jsonl"
RUN_ARCHIVE_BASE = RUN_OUTPUT_DIR / f"{RUN_ID}_benchmark_submission"
HISTORICAL_DIR = RUN_OUTPUT_DIR / "historical"
BUNDLE_ARGS_PATH = BENCHMARK_BUNDLE_DIR / "args.json"
BUNDLE_LOGGING_PATH = BENCHMARK_BUNDLE_DIR / "logging.jsonl"

INFERENCE_TARGET_CONFIG = {
    "mode": "remote_full_model",
    "base_model": DEFAULT_BASE_MODEL,
    "adapter": None,
    "model": MODEL_REPO,
    "use_hf": True,
    "label": None,
}

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
HISTORICAL_DIR.mkdir(parents=True, exist_ok=True)

environment = collect_environment_snapshot({**os.environ, **SWIFT_ENV})
run_info = {
    "run_id": RUN_ID,
    "workflow_type": "inference_only_backfill",
    "source_model_repo": MODEL_REPO,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "benchmark_bundle_dir": str(BENCHMARK_BUNDLE_DIR),
    "inference_requested_at_utc": utc_now_iso(),
    "inference_requested_at_israel": israel_now_iso(),
}

print(f"Run output dir: {RUN_OUTPUT_DIR}")
print(f"Bundle dir: {BENCHMARK_BUNDLE_DIR}")
print(json.dumps({"run": run_info, "environment": environment}, ensure_ascii=False, indent=2))


In [ ]:
HISTORICAL_ARGS_SOURCE = f"{MODEL_REPO}/args.json"
HISTORICAL_ARGS_PATH = copy_hf_file(MODEL_REPO, "args.json", HISTORICAL_DIR / "args.json", token=HF_TOKEN, required=True)
HISTORICAL_TRAINING_ARGS = json.loads(Path(HISTORICAL_ARGS_PATH).read_text(encoding="utf-8"))

BASE_MODEL = HISTORICAL_TRAINING_ARGS.get("model") or DEFAULT_BASE_MODEL
DATASET_ID = first_or_value(HISTORICAL_TRAINING_ARGS.get("dataset"))
VAL_DATASET_ID = first_or_value(HISTORICAL_TRAINING_ARGS.get("val_dataset"))

if not VAL_DATASET_ID:
    raise RuntimeError("args.json is missing val_dataset, which is required for dataset inference")

HISTORICAL_LOGGING_SOURCE = None
HISTORICAL_LOGGING_PATH = copy_hf_file(MODEL_REPO, "logging.jsonl", HISTORICAL_DIR / "logging.jsonl", token=HF_TOKEN, required=False)
if HISTORICAL_LOGGING_PATH is not None:
    HISTORICAL_LOGGING_SOURCE = f"{MODEL_REPO}/logging.jsonl"
    print(f"Using logging.jsonl from Hugging Face: {HISTORICAL_LOGGING_PATH}")
else:
    print("logging.jsonl was not found in the model repo. Run the next cell to upload the historical file from your training machine.")

INFER_CONFIG = {
    "enable_thinking": bool(HISTORICAL_TRAINING_ARGS.get("enable_thinking", False)),
    "add_non_thinking_prefix": bool(HISTORICAL_TRAINING_ARGS.get("add_non_thinking_prefix", True)),
    "temperature": HISTORICAL_TRAINING_ARGS.get("temperature", 0.0),
    "max_new_tokens": INFERENCE_MAX_NEW_TOKENS,
    "per_device_eval_batch_size": INFERENCE_BATCH_SIZE,
    "torch_dtype": HISTORICAL_TRAINING_ARGS.get("torch_dtype", "bfloat16"),
    "max_pixels": HISTORICAL_TRAINING_ARGS.get("max_pixels") or SWIFT_ENV["MAX_PIXELS"],
}

write_json(RUN_OUTPUT_DIR / "run_info.json", {
    "historical_args_source": HISTORICAL_ARGS_SOURCE,
    "historical_logging_source": HISTORICAL_LOGGING_SOURCE,
    "training_args": HISTORICAL_TRAINING_ARGS,
    "environment": environment,
    "run": {**run_info, "base_model": BASE_MODEL, "dataset": DATASET_ID, "validation_dataset": VAL_DATASET_ID},
})

print(json.dumps({
    "model_repo": MODEL_REPO,
    "base_model": BASE_MODEL,
    "dataset": DATASET_ID,
    "validation_dataset": VAL_DATASET_ID,
    "logging_source": HISTORICAL_LOGGING_SOURCE,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
}, ensure_ascii=False, indent=2))


In [ ]:
# Run this cell only if logging.jsonl was not downloaded from the model repo.
if HISTORICAL_LOGGING_PATH is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("This upload step is intended for Colab. Provide logging.jsonl another way if you are not on Colab.") from exc
    print("Upload the historical logging.jsonl from the training machine.")
    uploaded = files.upload()
    selected_name = next((name for name in uploaded if name.endswith(".jsonl")), None)
    if selected_name is None:
        raise RuntimeError("Expected a logging.jsonl upload")
    uploaded_path = Path(selected_name)
    destination = HISTORICAL_DIR / "logging.jsonl"
    shutil.move(str(uploaded_path), destination)
    HISTORICAL_LOGGING_PATH = destination
    HISTORICAL_LOGGING_SOURCE = "colab_upload"
else:
    print(f"Skipping upload because logging.jsonl is already available at {HISTORICAL_LOGGING_PATH}")

print(json.dumps({
    "historical_logging_path": None if HISTORICAL_LOGGING_PATH is None else str(HISTORICAL_LOGGING_PATH),
    "historical_logging_source": HISTORICAL_LOGGING_SOURCE,
}, ensure_ascii=False, indent=2))


In [ ]:
if HISTORICAL_LOGGING_PATH is None:
    raise RuntimeError("logging.jsonl is required for the benchmark bundle. Download it from the model repo or upload it in the previous cell.")

resolved_inference_target = resolve_inference_target(INFERENCE_TARGET_CONFIG, base_model=BASE_MODEL)

selected_checkpoint = {
    "checkpoint_name": resolved_inference_target["label"],
    "checkpoint_path": resolved_inference_target["adapter_ref"] or resolved_inference_target["model_ref"],
    "selection_metric": "preselected_model",
    "selection_reason": "existing_trained_model",
    "adapter_only": resolved_inference_target["adapter_ref"] is not None,
    "model_artifact_type": "adapter" if resolved_inference_target["adapter_ref"] is not None else "full_model",
    "evaluation_target_mode": resolved_inference_target["mode"],
    "evaluation_target_label": resolved_inference_target["label"],
    "evaluation_target_model_ref": resolved_inference_target["model_ref"],
    "evaluation_target_adapter_ref": resolved_inference_target["adapter_ref"],
    "evaluation_target_use_hf": resolved_inference_target["use_hf"],
    "historical_args_source": HISTORICAL_ARGS_SOURCE,
    "historical_logging_source": HISTORICAL_LOGGING_SOURCE,
}

infer_config = dict(INFER_CONFIG)

run_info.update({
    "base_model": BASE_MODEL,
    "dataset": DATASET_ID,
    "validation_dataset": VAL_DATASET_ID,
    "historical_args_source": HISTORICAL_ARGS_SOURCE,
    "historical_logging_source": HISTORICAL_LOGGING_SOURCE,
    "inference_target": resolved_inference_target,
    "inference_started_at_utc": utc_now_iso(),
    "inference_started_at_israel": israel_now_iso(),
})
write_json(RUN_OUTPUT_DIR / "run_info.json", {
    "training_args": HISTORICAL_TRAINING_ARGS,
    "environment": environment,
    "run": run_info,
    "selected_checkpoint": selected_checkpoint,
})

infer_started = time.time()
written_predictions = run_transformers_dataset_inference(
    resolved_inference_target,
    VAL_DATASET_ID,
    RAW_PREDICTIONS_PATH,
    PREDICTIONS_DIR,
    infer_config,
)
infer_finished = time.time()

run_info.update({
    "inference_finished_at_utc": utc_now_iso(),
    "inference_finished_at_israel": israel_now_iso(),
    "inference_duration_seconds": round(infer_finished - infer_started, 2),
})

model_metadata = build_benchmark_model_metadata(
    HISTORICAL_TRAINING_ARGS,
    environment,
    resolved_inference_target["label"],
    metadata_overrides={
        "model": resolved_inference_target["model_ref"],
        "use_hf": resolved_inference_target["use_hf"],
        "dataset": DATASET_ID,
        "validation_dataset": VAL_DATASET_ID,
        "per_device_eval_batch_size": INFERENCE_BATCH_SIZE,
    },
)

artifacts = {
    "args_json": "args.json",
    "logging_jsonl": "logging.jsonl",
    "prediction_dir": "predictions",
    "raw_predictions_jsonl": "raw_predictions.jsonl",
    "submission_archive": f"{RUN_ARCHIVE_BASE.name}.zip",
    "evaluation_target_mode": resolved_inference_target["mode"],
}

info_payload = build_submission_info_payload(
    model_metadata=model_metadata,
    training_args=HISTORICAL_TRAINING_ARGS,
    environment=environment,
    run=run_info,
    selected_checkpoint=selected_checkpoint,
    artifacts=artifacts,
)

write_json(BENCHMARK_BUNDLE_DIR / "info.json", info_payload)
shutil.copy2(HISTORICAL_ARGS_PATH, BUNDLE_ARGS_PATH)
shutil.copy2(HISTORICAL_LOGGING_PATH, BUNDLE_LOGGING_PATH)

instructions = """
This folder is a benchmark submission bundle generated on Colab.

On the benchmark machine:
1. Download and unzip the archive.
2. Point benchmark.input_dir at this folder, or copy its contents into the benchmark input folder.
3. Make sure the benchmark YAML mappings reference prediction files such as predictions/pred_0001.json.
4. Open the benchmark UI. If info.json is present, the UI should ingest it without manual form entry.
5. logging.jsonl in this bundle is the historical training log for the model.
""".strip()
(BENCHMARK_BUNDLE_DIR / "README.txt").write_text(instructions + "\n", encoding="utf-8")

archive_path = Path(shutil.make_archive(str(RUN_ARCHIVE_BASE), "zip", root_dir=RUN_OUTPUT_DIR, base_dir="benchmark_submission"))

summary = {
    "run_id": RUN_ID,
    "evaluation_target": resolved_inference_target,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
    "prediction_files": [path.name for path in written_predictions],
    "bundle_dir": str(BENCHMARK_BUNDLE_DIR),
    "archive_path": str(archive_path),
    "archive_name": archive_path.name,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


## What The User Downloads

Download the generated zip archive from Colab. It contains:

- `info.json`
- `args.json`
- `logging.jsonl`
- `predictions/pred_0001.json`, `predictions/pred_0002.json`, ...
- `README.txt`

That bundle can then be moved to the separate benchmark/UI machine.
